In [1]:
import os 
from dotenv import load_dotenv 

load_dotenv()

True

In [2]:
os.chdir('..')

In [3]:
data_path = os.path.join("data")
text_file_path = os.path.join(data_path,"AgenticAI.txt")
print(text_file_path)

data\AgenticAI.txt


In [4]:
from langchain_community.document_loaders import TextLoader 

loader = TextLoader(text_file_path)
documents = loader.load()

c:\Users\rsurs\OneDrive\Documents\multi-doc-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
documents[0].page_content[:500]

'Agentic AI represents the next major evolution in artificial intelligence, moving beyond traditional passive AI systems toward autonomous systems capable of planning, reasoning, and taking actions to achieve specific goals. Unlike conventional AI models that mainly respond to prompts or perform predefined tasks, agentic AI systems behave more like intelligent assistants that can independently decide what steps to take in order to complete a task.\n\nAt its core, agentic AI refers to AI systems des'

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)
text_chunks = splitter.split_documents(documents)

In [7]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5252.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS is a high performance vector search library. It only stores vectors, performs similarity search and index optimization. It doesn't manage `Documents`, `Metadata`,`Collection`,`Versions` 

In [8]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents=text_chunks,
                                    embedding=embeddings)

In [9]:
vector_store

In [10]:
retriever = vector_store.as_retriever()

In [11]:
query = "What is Agentic AI?"
docs = vector_store.similarity_search(query,k=4)

for i,doc in enumerate(docs):
    print(f"Document: {i + 1}")
    print(doc.page_content)
    print("-"*50)

Document: 1
At its core, agentic AI refers to AI systems designed as **agents**. An AI agent is a system that can perceive information from its environment, reason about it, make decisions, and take actions
--------------------------------------------------
Document: 2
Agentic AI represents the next major evolution in artificial intelligence, moving beyond traditional passive AI systems toward autonomous systems capable of planning, reasoning, and taking actions to
--------------------------------------------------
Document: 3
In conclusion, agentic AI represents a shift from reactive AI toward proactive and autonomous intelligence. By combining reasoning, planning, memory, and tool usage, agentic systems are becoming
--------------------------------------------------
Document: 4
agentic AI highly valuable in domains such as software development, healthcare, finance, and customer support.
--------------------------------------------------


In [12]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say you don't know. Use ten sentences maximum and keep answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [13]:
prompt = ChatPromptTemplate.from_template(template)

In [14]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [15]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [16]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context":retriever,"question":RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

rag_chain.invoke("What is agentic ai?")

'Agentic AI refers to AI systems designed as agents that can perceive information, reason, make decisions, and take actions. It represents the next major evolution in artificial intelligence, moving beyond traditional passive AI systems. Agentic AI systems are autonomous, capable of planning, reasoning, and taking actions. They combine reasoning, planning, memory, and tool usage to become proactive and autonomous. This type of AI is highly valuable in domains such as software development, healthcare, finance, and customer support. It shifts the focus from reactive AI to proactive and autonomous intelligence. Agentic AI agents can perceive their environment and make decisions based on that information. Overall, agentic AI enables the creation of more advanced and autonomous AI systems. It has the potential to transform various industries with its capabilities.'